# Experiment: First Quote Response Capture Gate

Objective:
- Model safe public labels for future lab quote replies.
- Block raw replies, private data, patient samples, and clinical/procurement routes.
- Feed only public-safe labels into the June 10 acceptance gate.

In [ ]:
from __future__ import annotations

PENDING = "quote_response_pending"
PASS = "quote_response_acceptance_ready"
HOLD = "quote_response_partial_endpoint_hold"
REJECT = "quote_response_rejected"
RELEASE_BLOCKED = "quote_response_release_blocked"

ACCEPTANCE_FIELDS = [
    "recipient_class_passed",
    "model_relevance",
    "controls",
    "hbf_or_hbg",
    "maturation",
    "viability",
    "alpha_or_autophagy",
    "hemolysis_or_membrane_safety",
    "raw_data",
    "cost",
    "timeline",
    "ethics",
    "biosafety",
    "material_constraints",
]

BEYOND_HBF_FIELDS = [
    "maturation",
    "viability",
    "alpha_or_autophagy",
    "hemolysis_or_membrane_safety",
]

RELEASE_BLOCK_FIELDS = [
    "raw_reply_text",
    "private_contact_details",
    "patient_identifiers",
    "patient_samples",
    "treatment_selection",
    "trial_screening",
    "purchase",
    "import",
    "procurement",
]

## Gate Logic

Public release is stricter than scientific acceptance. A reply can be scientifically
useful but still blocked from Git if it carries private content.

In [ ]:
def classify_captured_reply(reply: dict[str, bool]) -> dict[str, object]:
    """Classify a synthetic lab reply after private-only capture."""
    if not reply.get("reply_received"):
        return {"decision": PENDING, "reason": "no_reply", "fields": []}

    release_blocks = [field for field in RELEASE_BLOCK_FIELDS if reply.get(field)]
    if release_blocks:
        return {
            "decision": RELEASE_BLOCKED,
            "reason": "private_or_blocked_route",
            "fields": release_blocks,
        }

    if not reply.get("hbf_or_hbg"):
        return {
            "decision": REJECT,
            "reason": "missing_hbf_or_hbg",
            "fields": ["hbf_or_hbg"],
        }

    has_biology_beyond_hbf = any(reply.get(field) for field in BEYOND_HBF_FIELDS)
    if not has_biology_beyond_hbf:
        return {
            "decision": REJECT,
            "reason": "hbf_only_reply",
            "fields": BEYOND_HBF_FIELDS,
        }

    missing = [field for field in ACCEPTANCE_FIELDS if not reply.get(field)]
    if missing:
        return {
            "decision": HOLD,
            "reason": "missing_acceptance_fields",
            "fields": missing,
        }

    return {"decision": PASS, "reason": "public_safe_complete", "fields": []}

## Synthetic Reply Cases

These are labels, not real lab replies. Raw reply text is intentionally absent.

In [ ]:
complete_reply = {field: True for field in ACCEPTANCE_FIELDS}
complete_reply.update({field: False for field in RELEASE_BLOCK_FIELDS})
complete_reply["reply_received"] = True

cases = {
    "no_reply": complete_reply | {"reply_received": False},
    "complete_public_safe": complete_reply,
    "partial_endpoint": complete_reply
    | {
        "alpha_or_autophagy": False,
        "hemolysis_or_membrane_safety": False,
    },
    "hbf_only": complete_reply
    | {
        "maturation": False,
        "viability": False,
        "alpha_or_autophagy": False,
        "hemolysis_or_membrane_safety": False,
    },
    "raw_private_reply": complete_reply | {"raw_reply_text": True},
    "patient_sample_request": complete_reply | {"patient_samples": True},
}

results = {name: classify_captured_reply(reply) for name, reply in cases.items()}
results

In [ ]:
expected = {
    "no_reply": PENDING,
    "complete_public_safe": PASS,
    "partial_endpoint": HOLD,
    "hbf_only": REJECT,
    "raw_private_reply": RELEASE_BLOCKED,
    "patient_sample_request": RELEASE_BLOCKED,
}

decision_summary = {name: result["decision"] for name, result in results.items()}
assert decision_summary == expected, decision_summary
decision_summary

## Decision

- Continue only with label-level response capture.
- Keep raw replies and private details outside Git.
- Apply the June 10 acceptance gate only after release-blocking fields pass.